# Model Evaluation

Comprehensive evaluation of LLM, spaCy NER, and Rule-based extractors.

In [ ]:
import sys
sys.path.append('..')
import json
import asyncio
from evaluation.evaluator import ModelEvaluator
from src.core.llm_extractor import LLMExtractor
from src.core.rule_based_extractor import RuleBasedExtractor
from src.core.spacy_extractor import SpacyNERExtractor
import pandas as pd

## Load Ground Truth

In [ ]:
with open('../data/annotations/ground_truth_500.json', 'r') as f:
    ground_truth = json.load(f)
print(f'Loaded {len(ground_truth)} annotated samples')

## Run All Extractors

In [ ]:
async def evaluate_extractor(extractor, name):
    predictions = []
    for idx, sample in enumerate(ground_truth):
        if idx % 50 == 0:
            print(f'  Processing {idx}/{len(ground_truth)}...')
        result = await extractor.extract(sample)
        predictions.append(result)
    return predictions

print('Running LLM extractor...')
llm_predictions = await evaluate_extractor(LLMExtractor(prompt_version='v4'), 'LLM')

print('Running spaCy extractor...')
spacy_predictions = await evaluate_extractor(SpacyNERExtractor(), 'spaCy')

print('Running rule-based extractor...')
rule_predictions = await evaluate_extractor(RuleBasedExtractor(), 'Rule-based')

print('✓ All extractors complete')

## Compute Metrics

In [ ]:
evaluator = ModelEvaluator()

llm_results = evaluator.full_evaluation(llm_predictions, ground_truth)
spacy_results = evaluator.full_evaluation(spacy_predictions, ground_truth)
rule_results = evaluator.full_evaluation(rule_predictions, ground_truth)

print('LLM Results:', llm_results['classification_metrics'])
print('spaCy Results:', spacy_results['classification_metrics'])
print('Rule-based Results:', rule_results['classification_metrics'])

## Save Results

In [ ]:
evaluator.save_results(llm_results, 'experiment_results/llm_evaluation.json')
evaluator.save_results(spacy_results, 'experiment_results/spacy_evaluation.json')
evaluator.save_results(rule_results, 'experiment_results/rule_based_evaluation.json')
print('✓ Results saved')